# 05 Turkish Names


In [ ]:
from urllib.request import urlopen
import csv
import io
import unicodedata

import torch
import torch.nn.functional as F

url = "https://raw.githubusercontent.com/niyazikemer/turkce_isimler/main/turkce_isim.csv"
text = urlopen(url).read().decode("utf-8-sig")
rows = csv.DictReader(io.StringIO(text))
words = sorted(set(unicodedata.normalize("NFC", row["name"].strip().lower()) for row in rows))
words = [w for w in words if w and w.isalpha()]

chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(stoi)

print(len(words))
print(chars)
print([c for c in "çğıöşü" if c in stoi])

N = torch.zeros((vocab_size, vocab_size), dtype=torch.int32)
xs, ys = [], []

for w in words:
    chs = ["."] + list(w) + ["."]
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1
        xs.append(ix1)
        ys.append(ix2)

P = (N + 1).float()
P /= P.sum(1, keepdim=True)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
count_loss = -P[xs, ys].log().mean()

g = torch.Generator().manual_seed(42)
for i in range(10):
    out = []
    ix = 0
    while True:
        ix = torch.multinomial(P[ix], 1, replacement=True, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print("".join(out))

xenc = F.one_hot(xs, num_classes=vocab_size).float()
W = torch.randn((vocab_size, vocab_size), generator=g, requires_grad=True)

for k in range(200):
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    data_loss = -probs[torch.arange(len(ys)), ys].log().mean()
    loss = data_loss + 0.01 * (W**2).mean()
    W.grad = None
    loss.backward()
    W.data += -50 * W.grad

print("count model:", count_loss.item())
print("neural network:", data_loss.item())

P_nn = W.softmax(1)
for i in range(10):
    out = []
    ix = 0
    while True:
        ix = torch.multinomial(P_nn[ix], 1, replacement=True, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print("".join(out))
